# Full Forecasting Workflow

This notebook demonstrates a **complete end-to-end forecasting pipeline** using forecastbox.
We walk through every stage of a professional macro forecasting exercise:

1. **Problem Definition** — define the target variable, horizon, and data
2. **Auto-Forecast** — generate individual model forecasts automatically
3. **Baseline Comparison** — benchmark against naive and simple methods
4. **Forecast Combination** — combine models to improve accuracy
5. **Formal Evaluation** — apply statistical tests (DM, MCS, Mincer-Zarnowitz)
6. **Scenario Analysis** — build conditional forecasts under alternative policy paths
7. **Final Report** — consolidate results into a dashboard with fan chart

**Target**: Brazilian inflation (monthly), 12-month-ahead forecast.

**Datasets used**: `macro_brazil.csv` (Phase 1), `us_macro_quarterly.csv` (Phase 5)

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# forecastbox modules
from forecastbox.auto import AutoARIMA, AutoETS, AutoSelect
from forecastbox.combination import SimpleCombiner, WeightedCombiner, OLSCombiner
from forecastbox.evaluation import diebold_mariano, model_confidence_set, mincer_zarnowitz
from forecastbox.scenarios import SimpleVAR, ConditionalForecast, ScenarioBuilder, MonteCarlo, FanChart
from forecastbox.metrics import mae, rmse, mape, mase
from forecastbox.cv import expanding_window_cv

# Helpers
sys.path.insert(0, "..")
from utils.helpers import load_all_datasets

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("All forecastbox modules loaded successfully.")

## 1. Problem Definition

**Objective**: Forecast Brazilian monthly inflation 12 months ahead.

We start with exploratory data analysis — time series plot, autocorrelation function (ACF),
and seasonal decomposition — to understand the key features of the series before modelling.

In [ ]:
# Load Brazilian macro dataset
datasets = load_all_datasets()
df_brazil = datasets["macro_brazil"]
print(f"Shape: {df_brazil.shape}")
print(f"Date range: {df_brazil.index[0]} to {df_brazil.index[-1]}")
print(f"Columns: {list(df_brazil.columns)}")
df_brazil.head()

In [ ]:
# Target: inflation
inflation = df_brazil["inflation"]
h = 12  # forecast horizon

# Train/test split: hold out last 12 months for evaluation
train = inflation.iloc[:-h]
test = inflation.iloc[-h:]
print(f"Train: {len(train)} obs ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} obs ({test.index[0].date()} to {test.index[-1].date()})")

# EDA: time series plot, ACF, seasonal pattern
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: time series
axes[0].plot(train.index, train.values, "steelblue", linewidth=1.2, label="Train")
axes[0].plot(test.index, test.values, "darkorange", linewidth=1.2, label="Test")
axes[0].axvline(test.index[0], color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Brazilian Inflation (Monthly)", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: ACF
plot_acf(train.dropna(), lags=36, ax=axes[1], alpha=0.05)
axes[1].set_title("Autocorrelation Function", fontsize=12)

# Plot 3: monthly box plot for seasonality
monthly = train.groupby(train.index.month)
axes[2].boxplot([monthly.get_group(m).values for m in range(1, 13)],
                labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
axes[2].set_title("Seasonal Pattern (by Month)", fontsize=12)
axes[2].grid(True, alpha=0.3)

fig.suptitle("Exploratory Data Analysis: Brazilian Inflation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Step 1: Auto-Forecast

We use forecastbox's automatic model selection to generate individual forecasts:

- **AutoARIMA** — searches over ARIMA(p,d,q)(P,D,Q)[m] orders using AICc
- **AutoETS** — selects the best exponential smoothing model (error, trend, seasonal)
- **AutoSelect** — runs both families with cross-validation and picks the best overall

In [ ]:
# AutoARIMA
auto_arima = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aicc")
arima_result = auto_arima.fit(train)
arima_fc = arima_result.forecast(h)
print(f"AutoARIMA selected: {arima_result}")

# AutoETS
auto_ets = AutoETS(seasonal_period=12, ic="aicc")
ets_result = auto_ets.fit(train)
ets_fc = ets_result.forecast(h)
print(f"AutoETS selected: {ets_result}")

# AutoSelect (cross-validation based)
auto_select = AutoSelect(
    families=["arima", "ets"],
    cv_type="expanding",
    cv_horizon=h,
    cv_step=3,
    metric="rmse",
)
select_result = auto_select.fit(train, m=12)
select_fc = select_result.forecast(h)
print(f"AutoSelect winner: {select_result}")

# Summary table
model_table = pd.DataFrame({
    "Model": ["AutoARIMA", "AutoETS", "AutoSelect"],
    "MAE": [
        mae(test.values, arima_fc.point),
        mae(test.values, ets_fc.point),
        mae(test.values, select_fc.point),
    ],
    "RMSE": [
        rmse(test.values, arima_fc.point),
        rmse(test.values, ets_fc.point),
        rmse(test.values, select_fc.point),
    ],
})
print("\n=== Auto-Forecast Results ===")
print(model_table.to_string(index=False))

## 3. Step 2: Baseline Comparison

No forecasting exercise is complete without baselines. We compare against:

- **Naive** — last observed value repeated forward
- **Seasonal Naive** — same month from the previous year
- **Simple Moving Average (SMA)** — rolling mean of the last 12 observations

We use expanding-window temporal cross-validation to get reliable out-of-sample metrics.

In [ ]:
# Baseline forecasts on test set
# Naive: repeat last value
naive_fc = np.full(h, train.iloc[-1])

# Seasonal naive: same month last year
snaive_fc = train.iloc[-12:].values

# SMA(12)
sma_fc = np.full(h, train.iloc[-12:].mean())

# Collect all forecasts for comparison
all_forecasts = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "AutoSelect": select_fc.point,
    "Naive": naive_fc,
    "Seasonal Naive": snaive_fc,
    "SMA(12)": sma_fc,
}

# Temporal CV for baselines
def naive_model(y):
    """Simple wrapper for naive forecast."""
    class _Naive:
        def __init__(self, y): self._last = y.iloc[-1]
        def forecast(self, h, **kw):
            from forecastbox.core import Forecast
            return Forecast(point=np.full(h, self._last), model_name="Naive", horizon=h)
    return _Naive(y)

def snaive_model(y):
    class _SNaive:
        def __init__(self, y): self._season = y.iloc[-12:].values
        def forecast(self, h, **kw):
            from forecastbox.core import Forecast
            fc = np.tile(self._season, (h // 12) + 1)[:h]
            return Forecast(point=fc, model_name="SNaive", horizon=h)
    return _SNaive(y)

cv_naive = expanding_window_cv(train, naive_model, initial_window=60, horizon=h, step=6)
cv_snaive = expanding_window_cv(train, snaive_model, initial_window=60, horizon=h, step=6)

# Metrics table: test set
actual = test.values
metrics_rows = []
for name, fc in all_forecasts.items():
    metrics_rows.append({
        "Model": name,
        "MAE": round(mae(actual, fc), 4),
        "RMSE": round(rmse(actual, fc), 4),
        "MAPE": round(mape(actual, fc), 4),
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE")
print("=== All Models: Test-Set Metrics ===")
print(metrics_df.to_string(index=False))

print(f"\nBaseline CV (Naive)   — mean RMSE: {cv_naive.errors['rmse'].mean():.4f}")
print(f"Baseline CV (SNaive)  — mean RMSE: {cv_snaive.errors['rmse'].mean():.4f}")

## 4. Step 3: Forecast Combination

Combining forecasts typically outperforms any single model (Timmermann 2006).
We apply three strategies:

- **Simple Average** — equal weights
- **Inverse MSE** — weight inversely proportional to past MSE
- **Granger-Ramanathan (OLS)** — regression-based weights (optimal under MSE loss)

In [ ]:
# Prepare training forecasts for combination fitting
# Use a validation window from the training set
val_size = 24
train_part = inflation.iloc[:-(h + val_size)]
val_part = inflation.iloc[-(h + val_size):-h]

# Generate in-sample forecasts on validation period
arima_val = AutoARIMA(seasonal=True, m=12, stepwise=True).fit(train_part).forecast(val_size)
ets_val = AutoETS(seasonal_period=12).fit(train_part).forecast(val_size)

fc_train_list = [arima_val.point, ets_val.point]
actual_val = val_part.values

# 1. Simple average
simple_comb = SimpleCombiner(method="mean")
simple_comb.fit(fc_train_list, actual_val)
combined_simple = simple_comb.combine([arima_fc, ets_fc])

# 2. Inverse MSE
weighted_comb = WeightedCombiner(method="inverse_mse")
weighted_comb.fit(fc_train_list, actual_val)
combined_weighted = weighted_comb.combine([arima_fc, ets_fc])

# 3. Granger-Ramanathan (OLS)
ols_comb = OLSCombiner(intercept=False, constrained=True)
ols_comb.fit(fc_train_list, actual_val)
combined_ols = ols_comb.combine([arima_fc, ets_fc])

# Weights summary
weights_df = pd.DataFrame({
    "Method": ["Simple Average", "Inverse MSE", "Granger-Ramanathan"],
    "w(ARIMA)": [
        0.5,
        round(weighted_comb.weights_[0], 4),
        round(ols_comb.weights_[0], 4),
    ],
    "w(ETS)": [
        0.5,
        round(weighted_comb.weights_[1], 4),
        round(ols_comb.weights_[1], 4),
    ],
    "RMSE": [
        round(rmse(actual, combined_simple.point), 4),
        round(rmse(actual, combined_weighted.point), 4),
        round(rmse(actual, combined_ols.point), 4),
    ],
})
print("=== Combination Weights & Metrics ===")
print(weights_df.to_string(index=False))

## 5. Step 4: Formal Evaluation

Statistical tests give us rigorous answers to key questions:

- **Diebold-Mariano (DM)** — is one forecast significantly better than another?
- **Model Confidence Set (MCS)** — which models belong to the "best" set?
- **Mincer-Zarnowitz (MZ)** — are the forecasts well-calibrated (unbiased, efficient)?

In [ ]:
# Diebold-Mariano pairwise matrix
selected_models = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "Combined(OLS)": combined_ols.point,
    "Naive": naive_fc,
}

model_names = list(selected_models.keys())
n_models = len(model_names)
dm_matrix = pd.DataFrame(np.nan, index=model_names, columns=model_names)

for i in range(n_models):
    for j in range(n_models):
        if i != j:
            result = diebold_mariano(
                actual, selected_models[model_names[i]],
                selected_models[model_names[j]], h=1, loss="mse"
            )
            dm_matrix.iloc[i, j] = round(result.pvalue, 4)

print("=== Diebold-Mariano p-value Matrix ===")
print("(Row model vs Column model — low p-value means row is significantly different)")
print(dm_matrix.to_string())

# Model Confidence Set
mcs_result = model_confidence_set(
    actual, selected_models, alpha=0.10, loss="mse",
    n_boot=5000, seed=42
)
print(f"\n=== Model Confidence Set (alpha=0.10) ===")
print(f"Included models: {mcs_result.included_models}")
print(f"Excluded models: {mcs_result.excluded_models}")
print(f"Elimination order: {mcs_result.elimination_order}")
print(f"p-values: {mcs_result.pvalues}")

# Mincer-Zarnowitz for each model
print("\n=== Mincer-Zarnowitz Calibration Tests ===")
for name, fc in selected_models.items():
    mz = mincer_zarnowitz(actual, fc)
    calibrated = "Yes" if mz.pvalue > 0.05 else "No"
    print(f"{name:20s}: alpha={mz.alpha:.4f}, beta={mz.beta:.4f}, "
          f"F={mz.f_statistic:.3f}, p={mz.pvalue:.4f}, R2={mz.r_squared:.4f} — Calibrated: {calibrated}")

## 6. Step 5: Scenario Analysis

We estimate a **VAR(2)** model with GDP growth, inflation, and interest rate to generate
conditional forecasts under two alternative monetary policy scenarios:

- **Hawkish**: interest rate rises by +200 bps over 12 months
- **Dovish**: interest rate falls by -100 bps over 12 months

In [ ]:
# VAR model with 3 macro variables
var_vars = ["gdp_growth", "inflation", "interest_rate"]
endog = df_brazil[var_vars].dropna().values
var_model = SimpleVAR(endog, p_order=2, var_names=var_vars)
steps = 12

print(f"VAR({var_model.p_order}) with {var_model.k_vars} variables, {endog.shape[0]} observations")

# Unconditional forecast (baseline)
cf = ConditionalForecast(var_model, method="analytic")
baseline = cf.forecast(steps=steps, conditions=None, n_draws=1000, seed=42)

# Hawkish scenario: interest rate rises +200bps gradually
last_rate = df_brazil["interest_rate"].iloc[-1]
hawkish_rates = [last_rate + (200 / 100) * (t + 1) / steps for t in range(steps)]

# Dovish scenario: interest rate falls -100bps gradually
dovish_rates = [last_rate - (100 / 100) * (t + 1) / steps for t in range(steps)]

# Build scenarios
builder = ScenarioBuilder(var_model)
builder.add_scenario("baseline",
                     {"interest_rate": baseline["interest_rate"].point.tolist()},
                     description="Unconditional baseline")
builder.add_scenario("hawkish",
                     {"interest_rate": hawkish_rates},
                     description="Tightening: +200bps over 12 months")
builder.add_scenario("dovish",
                     {"interest_rate": dovish_rates},
                     description="Easing: -100bps over 12 months")

scenario_results = builder.run(steps=steps, n_draws=1000, seed=42)

# Plot scenarios
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"baseline": "blue", "hawkish": "red", "dovish": "green"}
horizons = np.arange(1, steps + 1)

for idx, var in enumerate(var_vars):
    ax = axes[idx]
    for scen_name in ["baseline", "hawkish", "dovish"]:
        fc = scenario_results.get(scen_name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
                label=scen_name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[scen_name])
    ax.set_title(var.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Horizon (months)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Monetary Policy Scenarios: Hawkish (+200bps) vs Dovish (-100bps)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Step 6: Final Report

We consolidate all results into a **dashboard** that includes:
- Model ranking by RMSE
- Summary metrics table
- Fan chart of the best forecast with prediction intervals
- Scenario comparison overlay

In [ ]:
# === FINAL DASHBOARD ===
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- Panel 1: Model Ranking (bar chart) ---
ax = axes[0, 0]
all_models_final = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "AutoSelect": select_fc.point,
    "Naive": naive_fc,
    "SNaive": snaive_fc,
    "SMA(12)": sma_fc,
    "Comb(Mean)": combined_simple.point,
    "Comb(InvMSE)": combined_weighted.point,
    "Comb(GR)": combined_ols.point,
}
ranking = sorted(all_models_final.items(), key=lambda x: rmse(actual, x[1]))
names_ranked = [r[0] for r in ranking]
rmses_ranked = [round(rmse(actual, r[1]), 4) for r in ranking]
bar_colors = ["green" if r < np.median(rmses_ranked) else "salmon" for r in rmses_ranked]
ax.barh(names_ranked, rmses_ranked, color=bar_colors, edgecolor="white")
ax.set_xlabel("RMSE")
ax.set_title("Model Ranking by RMSE", fontsize=12, fontweight="bold")
ax.invert_yaxis()
for i, v in enumerate(rmses_ranked):
    ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

# --- Panel 2: Metrics Table ---
ax = axes[0, 1]
ax.axis("off")
table_data = []
for name, fc in ranking[:5]:  # top 5
    table_data.append([
        name,
        f"{mae(actual, fc):.4f}",
        f"{rmse(actual, fc):.4f}",
        f"{mape(actual, fc):.4f}",
    ])
table = ax.table(
    cellText=table_data,
    colLabels=["Model", "MAE", "RMSE", "MAPE"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)
ax.set_title("Top-5 Models: Summary Metrics", fontsize=12, fontweight="bold", pad=20)

# --- Panel 3: Fan Chart of Best Model ---
ax = axes[1, 0]
best_name, best_fc = ranking[0]
# Use Monte Carlo on VAR for fan chart
mc = MonteCarlo(var_model, n_paths=1000, seed=42, parametric=True)
mc.simulate(steps=steps)
fan = mc.fan_chart(variable="inflation")
fan.plot(ax=ax, title=f"Inflation Fan Chart (VAR Monte Carlo)",
         color="steelblue", history_periods=24)
ax.set_ylabel("Inflation")
ax.set_xlabel("Period")

# --- Panel 4: Scenario Comparison ---
ax = axes[1, 1]
for scen_name in ["baseline", "hawkish", "dovish"]:
    fc = scenario_results.get(scen_name, "inflation")
    ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
            label=scen_name.capitalize(), linewidth=2, markersize=4)
    if fc.lower_80 is not None:
        ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                        alpha=0.15, color=colors[scen_name])
ax.set_title("Inflation Under Policy Scenarios", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("Inflation")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.suptitle("=== FORECASTING DASHBOARD: Brazilian Inflation ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# MCS summary
print("\n=== Model Confidence Set ===")
print(f"Best models (alpha=0.10): {mcs_result.included_models}")
print(f"\nConclusion: The MCS retains {len(mcs_result.included_models)} model(s) "
      f"as statistically indistinguishable from the best.")

## Exercise 1 — SOLUTION: Full Workflow for US GDP Growth

Complete end-to-end forecasting pipeline for US GDP growth (quarterly):

1. Load `us_macro_quarterly.csv`, target `gdp_growth`, horizon=8, m=4
2. AutoARIMA + AutoETS + naive baselines
3. Simple average + Granger-Ramanathan combination
4. DM test + MCS + Mincer-Zarnowitz evaluation
5. Fed scenarios: hawkish (+150bps) vs dovish (-75bps)
6. Professional dashboard with consolidated results

In [ ]:
# ============================================================
# Exercise 1 — SOLUTION: Full Workflow for US GDP Growth
# ============================================================

# --- 1. Data Loading & EDA ---
us_macro = datasets["us_macro_quarterly"]
print(f"US Macro Quarterly — Shape: {us_macro.shape}")
print(f"Date range: {us_macro.index[0]} to {us_macro.index[-1]}")
print(f"Columns: {list(us_macro.columns)}")
print(us_macro.describe().round(4))

# Target: GDP growth, quarterly
gdp = us_macro["gdp_growth"]
h_us = 8  # 8-quarter-ahead
m_us = 4  # quarterly frequency

# Train/test split
train_us = gdp.iloc[:-h_us]
test_us = gdp.iloc[-h_us:]
actual_us = test_us.values
print(f"\nTrain: {len(train_us)} obs ({train_us.index[0]} to {train_us.index[-1]})")
print(f"Test:  {len(test_us)} obs ({test_us.index[0]} to {test_us.index[-1]})")

# EDA visualization
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_us.index, train_us.values, "steelblue", linewidth=1.2, label="Train")
axes[0].plot(test_us.index, test_us.values, "darkorange", linewidth=1.2, label="Test")
axes[0].axvline(test_us.index[0], color="red", linestyle="--", alpha=0.5)
axes[0].set_title("US GDP Growth (Quarterly)", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

plot_acf(train_us.dropna(), lags=20, ax=axes[1], alpha=0.05)
axes[1].set_title("Autocorrelation Function", fontsize=12)

quarterly = train_us.groupby(train_us.index.quarter)
axes[2].boxplot([quarterly.get_group(q).values for q in range(1, 5)],
                labels=["Q1", "Q2", "Q3", "Q4"])
axes[2].set_title("Seasonal Pattern (by Quarter)", fontsize=12)
axes[2].grid(True, alpha=0.3)

fig.suptitle("EDA: US GDP Growth", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Auto-Forecast ---

# AutoARIMA (quarterly, m=4)
auto_arima_us = AutoARIMA(seasonal=True, m=m_us, stepwise=True, ic="aicc")
arima_result_us = auto_arima_us.fit(train_us)
arima_fc_us = arima_result_us.forecast(h_us)
print(f"AutoARIMA selected: {arima_result_us}")

# AutoETS (quarterly)
auto_ets_us = AutoETS(seasonal_period=m_us, ic="aicc")
ets_result_us = auto_ets_us.fit(train_us)
ets_fc_us = ets_result_us.forecast(h_us)
print(f"AutoETS selected: {ets_result_us}")

# --- 3. Baselines ---

# Naive: last value
naive_fc_us = np.full(h_us, train_us.iloc[-1])

# Seasonal naive: same quarter last year (lag 4)
snaive_fc_us = train_us.iloc[-4:].values
snaive_fc_us = np.tile(snaive_fc_us, (h_us // 4) + 1)[:h_us]

# SMA(4)
sma_fc_us = np.full(h_us, train_us.iloc[-4:].mean())

all_fc_us = {
    "AutoARIMA": arima_fc_us.point,
    "AutoETS": ets_fc_us.point,
    "Naive": naive_fc_us,
    "SNaive(4)": snaive_fc_us,
    "SMA(4)": sma_fc_us,
}

# Metrics summary
print("\n=== US GDP Auto-Forecast & Baselines ===")
for name, fc in all_fc_us.items():
    print(f"{name:15s}: MAE={mae(actual_us, fc):.4f}, RMSE={rmse(actual_us, fc):.4f}, "
          f"MAPE={mape(actual_us, fc):.4f}")

In [ ]:
# --- 4. Forecast Combination ---

# Validation set for combination weight estimation
val_size_us = 12
train_part_us = gdp.iloc[:-(h_us + val_size_us)]
val_part_us = gdp.iloc[-(h_us + val_size_us):-h_us]

arima_val_us = AutoARIMA(seasonal=True, m=m_us, stepwise=True).fit(train_part_us).forecast(val_size_us)
ets_val_us = AutoETS(seasonal_period=m_us).fit(train_part_us).forecast(val_size_us)

fc_train_us = [arima_val_us.point, ets_val_us.point]
actual_val_us = val_part_us.values

# Simple average
simple_us = SimpleCombiner(method="mean")
simple_us.fit(fc_train_us, actual_val_us)
comb_simple_us = simple_us.combine([arima_fc_us, ets_fc_us])

# Granger-Ramanathan
gr_us = OLSCombiner(intercept=False, constrained=True)
gr_us.fit(fc_train_us, actual_val_us)
comb_gr_us = gr_us.combine([arima_fc_us, ets_fc_us])

print("=== US GDP Combination Weights ===")
print(f"Simple Average: w(ARIMA)=0.5000, w(ETS)=0.5000, "
      f"RMSE={rmse(actual_us, comb_simple_us.point):.4f}")
print(f"Granger-Ram.:   w(ARIMA)={gr_us.weights_[0]:.4f}, w(ETS)={gr_us.weights_[1]:.4f}, "
      f"RMSE={rmse(actual_us, comb_gr_us.point):.4f}")

In [ ]:
# --- 5. Formal Evaluation: DM + MCS + Mincer-Zarnowitz ---

eval_models_us = {
    "AutoARIMA": arima_fc_us.point,
    "AutoETS": ets_fc_us.point,
    "Combined(GR)": comb_gr_us.point,
    "Naive": naive_fc_us,
}

# DM pairwise matrix
names_us = list(eval_models_us.keys())
dm_us = pd.DataFrame(np.nan, index=names_us, columns=names_us)
for i in range(len(names_us)):
    for j in range(len(names_us)):
        if i != j:
            res = diebold_mariano(
                actual_us, eval_models_us[names_us[i]],
                eval_models_us[names_us[j]], h=1, loss="mse"
            )
            dm_us.iloc[i, j] = round(res.pvalue, 4)

print("=== DM p-value Matrix (US GDP) ===")
print(dm_us.to_string())

# Model Confidence Set
mcs_us = model_confidence_set(
    actual_us, eval_models_us, alpha=0.10, loss="mse",
    n_boot=5000, seed=42
)
print(f"\n=== MCS (alpha=0.10) ===")
print(f"Included: {mcs_us.included_models}")
print(f"Excluded: {mcs_us.excluded_models}")
print(f"Elimination order: {mcs_us.elimination_order}")

# Mincer-Zarnowitz
print("\n=== Mincer-Zarnowitz Calibration (US GDP) ===")
for name, fc in eval_models_us.items():
    mz = mincer_zarnowitz(actual_us, fc)
    calibrated = "Yes" if mz.pvalue > 0.05 else "No"
    print(f"{name:15s}: alpha={mz.alpha:.4f}, beta={mz.beta:.4f}, "
          f"F={mz.f_statistic:.3f}, p={mz.pvalue:.4f}, R2={mz.r_squared:.4f} — Calibrated: {calibrated}")

In [ ]:
# --- 6. Scenario Analysis: Fed Hawkish vs Dovish ---

# VAR model on US macro
var_vars_us = ["gdp_growth", "inflation", "fed_funds"]
endog_us = us_macro[var_vars_us].dropna().values
var_us = SimpleVAR(endog_us, p_order=2, var_names=var_vars_us)
steps_us = 8

print(f"VAR({var_us.p_order}) with {var_us.k_vars} variables, {endog_us.shape[0]} obs")

# Unconditional baseline
cf_us = ConditionalForecast(var_us, method="analytic")
baseline_us = cf_us.forecast(steps=steps_us, conditions=None, n_draws=1000, seed=42)

# Hawkish: +150bps gradually
last_ff = us_macro["fed_funds"].iloc[-1]
hawkish_ff = [last_ff + (150 / 100) * (t + 1) / steps_us for t in range(steps_us)]

# Dovish: -75bps gradually
dovish_ff = [last_ff - (75 / 100) * (t + 1) / steps_us for t in range(steps_us)]

builder_us = ScenarioBuilder(var_us)
builder_us.add_scenario("baseline",
                        {"fed_funds": baseline_us["fed_funds"].point.tolist()},
                        description="Unconditional baseline")
builder_us.add_scenario("hawkish",
                        {"fed_funds": hawkish_ff},
                        description="Tightening: +150bps over 8 quarters")
builder_us.add_scenario("dovish",
                        {"fed_funds": dovish_ff},
                        description="Easing: -75bps over 8 quarters")

scen_us = builder_us.run(steps=steps_us, n_draws=1000, seed=42)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_us = {"baseline": "blue", "hawkish": "red", "dovish": "green"}
horizons_us = np.arange(1, steps_us + 1)

for idx, var in enumerate(var_vars_us):
    ax = axes[idx]
    for sn in ["baseline", "hawkish", "dovish"]:
        fc = scen_us.get(sn, var)
        ax.plot(horizons_us, fc.point, "-o", color=colors_us[sn],
                label=sn.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons_us, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors_us[sn])
    ax.set_title(var.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("US GDP: Fed Policy Scenarios — Hawkish (+150bps) vs Dovish (-75bps)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 7. Final Dashboard: US GDP Growth ---

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Model Ranking
ax = axes[0, 0]
all_us_final = {
    "AutoARIMA": arima_fc_us.point,
    "AutoETS": ets_fc_us.point,
    "Naive": naive_fc_us,
    "SNaive(4)": snaive_fc_us,
    "SMA(4)": sma_fc_us,
    "Comb(Mean)": comb_simple_us.point,
    "Comb(GR)": comb_gr_us.point,
}
ranking_us = sorted(all_us_final.items(), key=lambda x: rmse(actual_us, x[1]))
names_r = [r[0] for r in ranking_us]
rmses_r = [round(rmse(actual_us, r[1]), 4) for r in ranking_us]
bcolors = ["green" if r < np.median(rmses_r) else "salmon" for r in rmses_r]
ax.barh(names_r, rmses_r, color=bcolors, edgecolor="white")
ax.set_xlabel("RMSE")
ax.set_title("Model Ranking by RMSE", fontsize=12, fontweight="bold")
ax.invert_yaxis()
for i, v in enumerate(rmses_r):
    ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

# Panel 2: Metrics Table
ax = axes[0, 1]
ax.axis("off")
tdata = []
for name, fc in ranking_us[:5]:
    tdata.append([name, f"{mae(actual_us, fc):.4f}",
                  f"{rmse(actual_us, fc):.4f}", f"{mape(actual_us, fc):.4f}"])
tbl = ax.table(cellText=tdata,
               colLabels=["Model", "MAE", "RMSE", "MAPE"],
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 1.8)
ax.set_title("Top-5 Models: Summary Metrics", fontsize=12, fontweight="bold", pad=20)

# Panel 3: Fan Chart
ax = axes[1, 0]
mc_us = MonteCarlo(var_us, n_paths=1000, seed=42, parametric=True)
mc_us.simulate(steps=steps_us)
fan_us = mc_us.fan_chart(variable="gdp_growth")
fan_us.plot(ax=ax, title="GDP Growth Fan Chart (VAR Monte Carlo)",
            color="steelblue", history_periods=12)
ax.set_ylabel("GDP Growth (%)")
ax.set_xlabel("Period")

# Panel 4: Scenario Comparison
ax = axes[1, 1]
for sn in ["baseline", "hawkish", "dovish"]:
    fc = scen_us.get(sn, "gdp_growth")
    ax.plot(horizons_us, fc.point, "-o", color=colors_us[sn],
            label=sn.capitalize(), linewidth=2, markersize=4)
    if fc.lower_80 is not None:
        ax.fill_between(horizons_us, fc.lower_80, fc.upper_80,
                        alpha=0.15, color=colors_us[sn])
ax.set_title("GDP Under Fed Policy Scenarios", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.suptitle("=== FORECASTING DASHBOARD: US GDP Growth ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Reference values
print("\n=== Reference Values: US GDP Growth Workflow ===")
print(f"Best model: {ranking_us[0][0]} (RMSE={rmses_r[0]:.4f})")
print(f"MCS included: {mcs_us.included_models}")
print(f"GR weights: w(ARIMA)={gr_us.weights_[0]:.4f}, w(ETS)={gr_us.weights_[1]:.4f}")
print(f"Fed funds last: {last_ff:.2f}")
print(f"Hawkish terminal: {hawkish_ff[-1]:.2f}, Dovish terminal: {dovish_ff[-1]:.2f}")

### Interpretation: Exercise 1

**Key findings for US GDP Growth:**

1. **Auto-forecast**: AutoARIMA and AutoETS provide competitive individual forecasts.
   The quarterly frequency (m=4) captures the mild seasonality of US GDP.

2. **Combination**: The Granger-Ramanathan combination typically assigns larger weight
   to the model with lower validation-period MSE. The simple average serves as a robust
   benchmark — it often performs surprisingly well (Timmermann 2006).

3. **Evaluation**: The DM test identifies whether model differences are statistically
   significant. The MCS retains only those models that are statistically indistinguishable
   from the best. Mincer-Zarnowitz tests whether forecasts are unbiased and efficient.

4. **Scenarios**: Under hawkish tightening (+150bps), GDP growth declines as higher rates
   dampen demand. Under dovish easing (-75bps), growth is supported. The fan chart
   quantifies uncertainty around the baseline.

5. **Dashboard**: The consolidated view allows policymakers to see model rankings,
   metrics, uncertainty bands, and scenario impacts in a single professional display.

## Exercise 2 — SOLUTION: Nowcasting Step Before Forecasting

Extend the pipeline by adding a **DFM nowcasting step** before forecasting:

1. Use `mixed_freq.csv` with DFMNowcaster to estimate current-quarter GDP
2. Bridge the nowcast to the VAR as an initial condition
3. Run auto-forecast pipeline with and without the nowcast step
4. Compare accuracy: does the nowcast bridge improve forecasting performance?

In [ ]:
# ============================================================
# Exercise 2 — SOLUTION: Nowcasting + Forecasting Pipeline
# ============================================================

from forecastbox.nowcasting import DFMNowcaster

# --- 1. Load mixed-frequency data and fit DFM ---
mixed_freq = datasets["mixed_freq"]
print("=== Mixed-Frequency Dataset ===")
print(f"Shape: {mixed_freq.shape}")
print(f"Columns: {list(mixed_freq.columns)}")
print(f"Missing values:\n{mixed_freq.isna().sum()}")

# Define frequency map
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

# Fit DFM
dfm = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)
dfm.fit(mixed_freq)
print(f"\n{dfm}")

# Generate nowcast
nowcast_gdp = dfm.nowcast(target="gdp_growth")
nowcast_val = float(nowcast_gdp.point[0])
print(f"\n=== DFM GDP Nowcast ===")
print(f"Point: {nowcast_val:.4f}")
print(f"80% CI: [{nowcast_gdp.lower_80[0]:.4f}, {nowcast_gdp.upper_80[0]:.4f}]")
print(f"95% CI: [{nowcast_gdp.lower_95[0]:.4f}, {nowcast_gdp.upper_95[0]:.4f}]")

In [ ]:
# --- 2. VAR Forecasts: with vs without nowcast bridge ---

# VAR model on quarterly data
var_vars_ex2 = ["gdp_growth", "inflation", "fed_funds", "unemployment"]
endog_ex2 = us_macro[var_vars_ex2].dropna().values
var_ex2 = SimpleVAR(endog_ex2, p_order=2, var_names=var_vars_ex2)
steps_ex2 = 8

cf_ex2 = ConditionalForecast(var_ex2, method="analytic")

# WITHOUT nowcast (unconditional)
fc_no_nowcast = cf_ex2.forecast(steps=steps_ex2, conditions=None, n_draws=1000, seed=42)

# WITH nowcast bridge: condition Q1 GDP on DFM estimate
fc_with_nowcast = cf_ex2.forecast(
    steps=steps_ex2,
    conditions={"gdp_growth": [nowcast_val]},
    n_draws=1000,
    seed=42,
)

print("=== GDP Forecast Comparison (Q1) ===")
print(f"Without nowcast: {fc_no_nowcast['gdp_growth'].point[0]:.4f}")
print(f"With nowcast:    {fc_with_nowcast['gdp_growth'].point[0]:.4f}")
print(f"DFM nowcast:     {nowcast_val:.4f}")

In [ ]:
# --- 3. Auto-forecast with nowcast-augmented initial condition ---
# Re-run AutoARIMA/AutoETS on GDP, appending nowcast as the latest observation

# Create augmented GDP series with nowcast appended
gdp_train = us_macro["gdp_growth"].iloc[:-h_us]
last_date = gdp_train.index[-1]
nowcast_date = last_date + pd.tseries.offsets.QuarterEnd(1)
gdp_augmented = pd.concat([
    gdp_train,
    pd.Series([nowcast_val], index=[nowcast_date], name="gdp_growth")
])

print(f"Original train: {len(gdp_train)} obs, last={gdp_train.index[-1]}")
print(f"Augmented train: {len(gdp_augmented)} obs, last={gdp_augmented.index[-1]}")
print(f"Nowcast appended: {nowcast_val:.4f} at {nowcast_date}")

# Auto-forecast WITHOUT nowcast
arima_no = AutoARIMA(seasonal=True, m=m_us, stepwise=True, ic="aicc").fit(gdp_train)
fc_arima_no = arima_no.forecast(h_us)

ets_no = AutoETS(seasonal_period=m_us, ic="aicc").fit(gdp_train)
fc_ets_no = ets_no.forecast(h_us)

# Auto-forecast WITH nowcast augmentation
arima_aug = AutoARIMA(seasonal=True, m=m_us, stepwise=True, ic="aicc").fit(gdp_augmented)
fc_arima_aug = arima_aug.forecast(h_us)

ets_aug = AutoETS(seasonal_period=m_us, ic="aicc").fit(gdp_augmented)
fc_ets_aug = ets_aug.forecast(h_us)

# Combine: simple average
comb_no = SimpleCombiner(method="mean")
comb_no_fc = comb_no.combine([fc_arima_no, fc_ets_no])

comb_aug = SimpleCombiner(method="mean")
comb_aug_fc = comb_aug.combine([fc_arima_aug, fc_ets_aug])

print("\n=== Forecasts: Without vs With Nowcast ===")
print(f"ARIMA (no nowcast):  {fc_arima_no.point[:4].round(4)}")
print(f"ARIMA (w/ nowcast):  {fc_arima_aug.point[:4].round(4)}")
print(f"ETS (no nowcast):    {fc_ets_no.point[:4].round(4)}")
print(f"ETS (w/ nowcast):    {fc_ets_aug.point[:4].round(4)}")
print(f"Combined (no nc):    {comb_no_fc.point[:4].round(4)}")
print(f"Combined (w/ nc):    {comb_aug_fc.point[:4].round(4)}")

In [ ]:
# --- 4. Accuracy Comparison: with vs without nowcasting ---

comparison = {
    "ARIMA (no nowcast)": fc_arima_no.point,
    "ARIMA (w/ nowcast)": fc_arima_aug.point,
    "ETS (no nowcast)": fc_ets_no.point,
    "ETS (w/ nowcast)": fc_ets_aug.point,
    "Combined (no nc)": comb_no_fc.point,
    "Combined (w/ nc)": comb_aug_fc.point,
}

print("=== Accuracy: With vs Without Nowcasting ===")
comp_rows = []
for name, fc in comparison.items():
    comp_rows.append({
        "Pipeline": name,
        "MAE": round(mae(actual_us, fc), 4),
        "RMSE": round(rmse(actual_us, fc), 4),
        "MAPE": round(mape(actual_us, fc), 4),
    })
comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

# Improvement summary
rmse_no = rmse(actual_us, comb_no_fc.point)
rmse_aug = rmse(actual_us, comb_aug_fc.point)
improvement = (rmse_no - rmse_aug) / rmse_no * 100
print(f"\nRMSE improvement from nowcasting: {improvement:+.2f}%")
print(f"  Combined (no nowcast):  RMSE = {rmse_no:.4f}")
print(f"  Combined (w/ nowcast):  RMSE = {rmse_aug:.4f}")

In [ ]:
# --- 5. Dashboard: Nowcasting Impact ---

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Forecast paths comparison
ax = axes[0, 0]
horizons_nc = np.arange(1, h_us + 1)
ax.plot(horizons_nc, comb_no_fc.point, "b-o", label="Without Nowcast", linewidth=2)
ax.plot(horizons_nc, comb_aug_fc.point, "r-s", label="With Nowcast", linewidth=2)
ax.plot(horizons_nc, actual_us, "k--", label="Actual", linewidth=2, alpha=0.7)
ax.axhline(nowcast_val, color="red", linestyle=":", alpha=0.5, label=f"Nowcast={nowcast_val:.2f}")
ax.set_title("Combined Forecast: With vs Without Nowcast", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Error comparison
ax = axes[0, 1]
err_no = np.abs(actual_us - comb_no_fc.point)
err_aug = np.abs(actual_us - comb_aug_fc.point)
x_pos = np.arange(h_us)
width = 0.35
ax.bar(x_pos - width/2, err_no, width, label="Without Nowcast", color="steelblue", alpha=0.8)
ax.bar(x_pos + width/2, err_aug, width, label="With Nowcast", color="darkorange", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"Q+{i+1}" for i in range(h_us)])
ax.set_title("Absolute Errors by Horizon", fontsize=12, fontweight="bold")
ax.set_ylabel("|Error|")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: VAR bridged vs unconditional
ax = axes[1, 0]
horizons_var = np.arange(1, steps_ex2 + 1)
ax.plot(horizons_var, fc_no_nowcast["gdp_growth"].point, "b-o",
        label="VAR Unconditional", linewidth=2)
ax.plot(horizons_var, fc_with_nowcast["gdp_growth"].point, "r-s",
        label=f"VAR Bridged (nc={nowcast_val:.2f})", linewidth=2)
if fc_with_nowcast["gdp_growth"].lower_80 is not None:
    ax.fill_between(horizons_var,
                    fc_with_nowcast["gdp_growth"].lower_80,
                    fc_with_nowcast["gdp_growth"].upper_80,
                    alpha=0.15, color="red")
ax.axhline(nowcast_val, color="red", linestyle=":", alpha=0.5)
ax.set_title("VAR: Unconditional vs Nowcast Bridge", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 4: Metrics comparison table
ax = axes[1, 1]
ax.axis("off")
tbl_data = []
for name, fc in comparison.items():
    tbl_data.append([name, f"{mae(actual_us, fc):.4f}",
                     f"{rmse(actual_us, fc):.4f}", f"{mape(actual_us, fc):.4f}"])
tbl2 = ax.table(cellText=tbl_data,
                colLabels=["Pipeline", "MAE", "RMSE", "MAPE"],
                cellLoc="center", loc="center")
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(9)
tbl2.scale(1.0, 1.6)
ax.set_title("Accuracy: With vs Without Nowcasting", fontsize=12, fontweight="bold", pad=20)

fig.suptitle("=== NOWCASTING IMPACT DASHBOARD ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Reference values
print("\n=== Reference Values: Nowcasting Pipeline ===")
print(f"DFM Nowcast: {nowcast_val:.4f}")
print(f"RMSE without nowcast: {rmse_no:.4f}")
print(f"RMSE with nowcast: {rmse_aug:.4f}")
print(f"Improvement: {improvement:+.2f}%")
print(f"VAR GDP Q1 unconditional: {fc_no_nowcast['gdp_growth'].point[0]:.4f}")
print(f"VAR GDP Q1 bridged: {fc_with_nowcast['gdp_growth'].point[0]:.4f}")

### Interpretation: Exercise 2

**Key findings from the nowcasting pipeline:**

1. **DFM Nowcast**: The Dynamic Factor Model extracts a latent factor from high-frequency
   monthly indicators (industrial production, retail sales, confidence) to estimate
   current-quarter GDP before the official release.

2. **Bridge Effect**: Conditioning the VAR's first period on the DFM nowcast shifts the
   entire forecast trajectory. This is especially impactful when the nowcast differs
   substantially from the VAR's unconditional Q1 prediction.

3. **Accuracy Comparison**: The nowcast-augmented pipeline typically improves short-horizon
   accuracy (Q+1, Q+2) because it anchors the forecast to timely information. The benefit
   fades at longer horizons where the VAR dynamics dominate.

4. **Practical Value**: Central banks routinely use this bridge technique. The DFM provides
   a real-time estimate of the present, which feeds into the conditional forecast. This
   pipeline is the standard for monetary policy analysis.

5. **Reference**: The RMSE improvement quantifies the marginal value of nowcasting.
   Even modest improvements in short-horizon accuracy can be economically significant
   for policy decisions.